In [ ]:
# 1. Environment Setup & Constants
import os
import sys

def install_deps():
    print("Installing compatible packages...")
    !pip install -q -U "pandas<2.2.0" "numpy<2.0.0" sdv sdmetrics scipy scikit-learn matplotlib seaborn networkx
    print("\nRestarting runtime to apply changes...")
    os._exit(0)

try:
    import numpy as np
    import pandas as pd
    import sdv
    from sdv.metadata import Metadata
    from sdv.single_table import CTGANSynthesizer, GaussianCopulaSynthesizer
    from sdmetrics.reports.single_table import QualityReport
    if not np.version.version.startswith('1'):
        install_deps()
except (ImportError, ModuleNotFoundError):
    install_deps()

import random, shutil
from pathlib import Path
from scipy.stats import ks_2samp

RANDOM_STATE = 42
N_SYNTHETIC = 30
TABLE_NAME = "pcos_ecg"
OUTPUT_ROOT = Path("outputs")
SYNTH_DIR, REPORT_DIR, PLOT_DIR = OUTPUT_ROOT/"synthetic", OUTPUT_ROOT/"reports", OUTPUT_ROOT/"plots"

for d in [OUTPUT_ROOT, SYNTH_DIR, REPORT_DIR, PLOT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

def reset_seeds(seed=RANDOM_STATE):
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
reset_seeds()
print("Environment and constants initialized.")

# PCOS ECG-Centric Synthetic Data Generation and Validation

1. Audits the uploaded 10-subject PCOS dataset.
2. Trains CTGAN, Gaussian Copula, and a mixed-type Bayesian network.
3. Generates 30 rows from each.
4. Runs SDMetrics and statistical validation.

In [ ]:
# 0. Install specific compatible versions to avoid NumPy 2.0/SciPy conflicts
%pip install -q -U "pandas<2.2.0" "numpy<2.0.0" sdv sdmetrics scipy scikit-learn matplotlib seaborn networkx

In [ ]:
import os
import sys
import json
import math
import random
import shutil
import warnings
from pathlib import Path
from collections import defaultdict, deque

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx

from scipy.stats import ks_2samp, wasserstein_distance
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import normalized_mutual_info_score
from sklearn.linear_model import LinearRegression

from sdv.metadata import Metadata
from sdv.single_table import CTGANSynthesizer, GaussianCopulaSynthesizer
from sdmetrics.reports.single_table import QualityReport

warnings.filterwarnings("ignore", category=FutureWarning)

RANDOM_STATE = 42
N_SYNTHETIC = 30
TABLE_NAME = "pcos_ecg"

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
os.environ["PYTHONHASHSEED"] = str(RANDOM_STATE)

OUTPUT_ROOT = Path("outputs")
SYNTH_DIR = OUTPUT_ROOT / "synthetic_datasets"
REPORT_DIR = OUTPUT_ROOT / "reports"
PLOT_DIR = OUTPUT_ROOT / "plots"
COMPARISON_DIR = OUTPUT_ROOT / "comparison"

for directory in [OUTPUT_ROOT, SYNTH_DIR, REPORT_DIR, PLOT_DIR, COMPARISON_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", context="paper", font_scale=1.1)
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "font.family": "DejaVu Sans",
    "axes.titleweight": "bold",
})

print(f"Python: {sys.version.split()[0]}")
try:
    import sdv
    import sdmetrics
    print(f"SDV: {sdv.__version__}")
    print(f"SDMetrics: {sdmetrics.__version__}")
except Exception:
    pass

In [ ]:
# 0. Data Loading and Preprocessing
IDENTIFIER = "Subject_ID"
LABEL_COLUMNS = ["Diagnosis", "Target_Class"]
CATEGORICAL_COLUMNS = ["Diagnosis", "Target_Class", "Cycle Regularity", "Family History (PCOS/Diabetes)"]
NUMERICAL_FEATURES = ["HRV_SDNN_Estimate (ms)", "Mean_RR_Interval (s)", "QTc_Bazett (s)", "QT_Interval (s)", "P_Wave_Dispersion (s)", "PR_Interval (s)", "QRS_Duration (s)", "T_Wave_Duration (s)", "SDNN_MeanRR_Ratio", "QTc_MeanRR_Ratio", "PWaveDisp_x_QTc", "HRVSDNN_x_QTc", "Cycle Length", "BMI"]
EXPECTED_COLUMNS = [IDENTIFIER] + LABEL_COLUMNS + [c for c in NUMERICAL_FEATURES + CATEGORICAL_COLUMNS if c not in LABEL_COLUMNS]

def normalize_column_names(frame):
    result = frame.copy()
    result.columns = [str(c).strip().replace("\n", " ").replace("\r", " ") for c in result.columns]
    return result

def locate_input_csv():
    path = Path("/content/PCOS_ECG_Centric_16_Features_Corrected.csv")
    if path.exists(): return path
    return Path("PCOS_ECG_Centric_16_Features_Corrected.csv")

INPUT_PATH = locate_input_csv()
if not INPUT_PATH.exists():
    print(f"Error: {INPUT_PATH} not found. Please upload the CSV.")
else:
    raw_data = normalize_column_names(pd.read_csv(INPUT_PATH))
    data = raw_data[EXPECTED_COLUMNS].copy()

    # Preprocessing: Handle numeric conversion and imputation
    for column in NUMERICAL_FEATURES:
        data[column] = pd.to_numeric(data[column], errors="coerce")
        data[column] = data[column].fillna(data[column].median())

    for column in CATEGORICAL_COLUMNS:
        data[column] = data[column].astype("string").fillna("Missing")

    # Ensure unique IDs
    data[IDENTIFIER] = [f"SUBJ_{i:04d}" for i in range(1, len(data) + 1)]

    print("Data loaded successfully.")
    display(data.head())

In [ ]:
# 2. Metadata and Model Training
metadata = Metadata.detect_from_dataframe(data=data, table_name=TABLE_NAME)
metadata.update_column(column_name=IDENTIFIER, sdtype="id")
metadata.set_primary_key(column_name=IDENTIFIER)
for column in CATEGORICAL_COLUMNS: metadata.update_column(column_name=column, sdtype="categorical")
for column in NUMERICAL_FEATURES: metadata.update_column(column_name=column, sdtype="numerical")

# Train Models
ctgan = CTGANSynthesizer(metadata, epochs=500)
ctgan.fit(data)
gaussian = GaussianCopulaSynthesizer(metadata)
gaussian.fit(data)

# Generate Data
synthetic_datasets = {
    "CTGAN": ctgan.sample(N_SYNTHETIC),
    "GaussianCopula": gaussian.sample(N_SYNTHETIC)
}

# Save Files
for name, df in synthetic_datasets.items():
    df.to_csv(SYNTH_DIR / f"{name.lower()}_30.csv", index=False)
print("Models trained and data generated.")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sdmetrics.reports.single_table import QualityReport
from scipy.stats import ks_2samp

# 3. Validation and Visualization for ALL models
quality_rows = []
metrics_metadata = metadata.to_dict()['tables'][TABLE_NAME]
original_corr = data[NUMERICAL_FEATURES].corr(method='pearson')

for name, synthetic in synthetic_datasets.items():
    print(f"Processing results for: {name}")

    # Quality Report
    report = QualityReport()
    report.generate(data, synthetic, metrics_metadata, verbose=False)
    score = report.get_score()

    # Correlation Analysis
    synth_corr = synthetic[NUMERICAL_FEATURES].corr(method='pearson')
    diff = (original_corr - synth_corr).abs()
    mean_corr_diff = np.nanmean(diff.to_numpy()[np.triu_indices_from(diff, k=1)])

    # KS Statistics
    ks_stats = [ks_2samp(data[col], synthetic[col]).statistic for col in NUMERICAL_FEATURES]
    mean_ks = np.mean(ks_stats)

    quality_rows.append({
        'Synthesizer': name,
        'Overall Quality Score': score,
        'Mean Correlation Difference': mean_corr_diff,
        'Mean KS Statistic': mean_ks
    })

    # Generate and Save Heatmap
    plt.figure(figsize=(12, 10))
    sns.heatmap(synth_corr, cmap='coolwarm', center=0, annot=False)
    plt.title(f'Correlation Matrix: {name}')
    plt.tight_layout()
    plt.savefig(PLOT_DIR / f'{name.lower()}_correlation_heatmap.png')
    plt.show()
    plt.close()

comparison = pd.DataFrame(quality_rows).sort_values(by='Overall Quality Score', ascending=False)
display(comparison)
print(f'Best Synthesizer: {comparison.iloc[0]["Synthesizer"]}')

In [ ]:
# 4. Export results
import shutil
from pathlib import Path
from google.colab import files

# Ensure path constants are available
OUTPUT_ROOT = Path("outputs")

if OUTPUT_ROOT.exists():
    shutil.make_archive('PCOS_ECG_Synthetic_Results', 'zip', OUTPUT_ROOT)
    try:
        files.download('PCOS_ECG_Synthetic_Results.zip')
        print("Download started. Check your browser downloads.")
    except Exception as e:
        print(f'Download failed: {e}. The file is located at /content/PCOS_ECG_Synthetic_Results.zip')
else:
    print("Error: Output folder not found. Please run the training/validation cells first.")

### 5. Final Export of Evaluation Metrics
This cell saves the comparison table (the ranking of synthesizers) and provides a final download for all results.

In [ ]:
import shutil
from google.colab import files

# Final Export including all generated plots
comparison_file = OUTPUT_ROOT / 'final_synthesizer_comparison_report.csv'
comparison.to_csv(comparison_file, index=False)

# Create a fresh archive of the outputs folder containing all CSVs and PNGs
archive_path = shutil.make_archive('PCOS_ECG_All_Methodologies_Results', 'zip', OUTPUT_ROOT)

print(f"Final ranking saved to {comparison_file}")
try:
    files.download('PCOS_ECG_All_Methodologies_Results.zip')
    print("Download of the updated ZIP (with all 3 plots) has started.")
except Exception as e:
    print(f'Download failed: {e}. Archive is at /content/PCOS_ECG_All_Methodologies_Results.zip')

display(comparison)

In [ ]:
# 11. Unified Training and Data Generation
bayesian = BayesianNetworkSynthesizer(CATEGORICAL_COLUMNS, NUMERICAL_FEATURES, IDENTIFIER)
bayesian.fit(data)

ctgan = CTGANSynthesizer(metadata, epochs=50)
ctgan.fit(data)

gaussian = GaussianCopulaSynthesizer(metadata)
gaussian.fit(data)

# Populate the dictionary for comparison
synthetic_datasets = {
    "CTGAN": ctgan.sample(N_SYNTHETIC),
    "GaussianCopula": gaussian.sample(N_SYNTHETIC),
    "BayesianNetwork": bayesian.sample(N_SYNTHETIC)
}

for name, df in synthetic_datasets.items():
    df.to_csv(SYNTH_DIR / f"{name.lower()}_30.csv", index=False)
    print(f"Saved {name} synthetic data.")

In [ ]:
report = QualityReport()
report.generate(data, synthetic_datasets['CTGAN'], metadata.to_dict()['tables'][TABLE_NAME])
display(report.get_properties())

In [ ]:
import pandas as pd
import numpy as np
from sdmetrics.reports.single_table import QualityReport

# Ensure we have the latest results for all models in the dictionary
results = []
orig_corr = data[NUMERICAL_FEATURES].corr()
metrics_metadata = metadata.to_dict()['tables'][TABLE_NAME]

for name, synth in synthetic_datasets.items():
    # Quality Report
    report = QualityReport()
    report.generate(data, synth, metrics_metadata, verbose=False)

    # Manual Correlation Error
    synth_corr = synth[NUMERICAL_FEATURES].corr()
    corr_err = np.nanmean(np.abs(orig_corr - synth_corr).values)

    results.append({
        'Synthesizer': name,
        'Quality Score': report.get_score(),
        'Mean Corr Error': corr_err
    })

# Update the global comparison dataframe
comparison = pd.DataFrame(results).sort_values('Quality Score', ascending=False)

print('Final Comparison including Bayesian Network:')
display(comparison)
print(f'Best Synthesizer: {comparison.iloc[0]["Synthesizer"]}')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sdmetrics.reports.single_table import QualityReport
from scipy.stats import ks_2samp, wasserstein_distance
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# 13. Deep Clinical Validation: Metrics & Visualizations
final_metrics = []
scaler = StandardScaler()
data_scaled = scaler.fit_transform(data[NUMERICAL_FEATURES])

pca = PCA(n_components=2)
pca_orig = pca.fit_transform(data_scaled)

for name, synthetic in synthetic_datasets.items():
    print(f"Deep evaluation for: {name}")

    # A. SDV Detailed Report
    report = QualityReport()
    report.generate(data, synthetic, metrics_metadata, verbose=False)
    prop = report.get_properties()
    col_shapes = prop.loc[prop['Property'] == 'Column Shapes', 'Score'].values[0]
    col_trends = prop.loc[prop['Property'] == 'Column Pair Trends', 'Score'].values[0]

    # B. Statistical Distances
    ks_results = []
    wass_distances = []
    for col in NUMERICAL_FEATURES:
        ks_res = ks_2samp(data[col], synthetic[col])
        ks_results.append({'col': col, 'stat': ks_res.statistic, 'p': ks_res.pvalue})
        wass_distances.append(wasserstein_distance(data[col], synthetic[col]))

    avg_ks_stat = np.mean([x['stat'] for x in ks_results])
    avg_wass = np.mean(wass_distances)

    # C. Correlation Matrices
    orig_corr = data[NUMERICAL_FEATURES].corr()
    synth_corr = synthetic[NUMERICAL_FEATURES].corr()
    corr_diff = (orig_corr - synth_corr).abs()

    # Save Matrices
    synth_corr.to_csv(PLOT_DIR / f"{name.lower()}_corr_matrix.csv")
    corr_diff.to_csv(PLOT_DIR / f"{name.lower()}_corr_diff_matrix.csv")

    # D. PCA Visualization
    synth_scaled = scaler.transform(synthetic[NUMERICAL_FEATURES])
    pca_synth = pca.transform(synth_scaled)

    plt.figure(figsize=(10, 6))
    plt.scatter(pca_orig[:, 0], pca_orig[:, 1], alpha=0.5, label='Original', c='blue')
    plt.scatter(pca_synth[:, 0], pca_synth[:, 1], alpha=0.5, label=name, c='red', marker='x')
    plt.title(f'PCA Comparison: Original vs {name}')
    plt.legend()
    plt.savefig(PLOT_DIR / f"{name.lower()}_pca_plot.png")
    plt.show()
    plt.close()

    final_metrics.append({
        'Synthesizer': name,
        'Overall Quality Score': report.get_score(),
        'Column Shapes Score': col_shapes,
        'Column Pair Trends Score': col_trends,
        'Mean Correlation Difference': corr_diff.values.mean(),
        'Mean KS Statistic': avg_ks_stat,
        'Mean Wasserstein Distance': avg_wass
    })

final_comparison_df = pd.DataFrame(final_metrics).sort_values(by='Overall Quality Score', ascending=False)
display(final_comparison_df)

# Determine best and save to file
best_model = final_comparison_df.iloc[0]['Synthesizer']
with open(OUTPUT_ROOT / "best_synthesizer.txt", "w") as f:
    f.write(best_model)

final_comparison_df.to_csv(OUTPUT_ROOT / "comprehensive_comparison_report.csv", index=False)

In [ ]:
# 14. Final ZIP Export with all requirements
shutil.make_archive('PCOS_Clinical_Validation_Package', 'zip', OUTPUT_ROOT)
try:
    files.download('PCOS_Clinical_Validation_Package.zip')
    print("Download of full validation package initiated.")
except Exception as e:
    print(f"File available at /content/PCOS_Clinical_Validation_Package.zip")

In [ ]:
# 12. Updated final ranking including Bayesian Network
final_ranking = comparison.sort_values(by=['Quality Score', 'Mean Corr Error'], ascending=[False, True])

print('Final Synthesizer Ranking (All Models):')
display(final_ranking)

winner = final_ranking.iloc[0]['Synthesizer']
print(f'\nBest Overall Synthesizer: {winner}')

# Save and export the updated results
comparison_file = OUTPUT_ROOT / 'final_synthesizer_comparison.csv'
final_ranking.to_csv(comparison_file, index=False)
shutil.make_archive('PCOS_ECG_Full_Evaluation_Results', 'zip', OUTPUT_ROOT)
print(f'\nUpdated results exported to PCOS_ECG_Full_Evaluation_Results.zip')